# Unix 2A: streams, pipelines, and text processing

This notebook covers stdin, stdout, stderr, redirection, pipelines, and text filters. Work through it before the Bash scripting notebook.


## Learning outcomes

By the end of the workshops, you should be able to:

- distinguish stdin, stdout, and stderr;
- connect commands with pipes and redirect output safely;
- choose delimiters and sort keys that match the data;
- combine `grep`, `sort`, `cut`, `awk`, and `sed`;
- write a Bash script with quoted variables and arguments;
- iterate safely over files and lines; and
- use conditions and exit status to respond to errors.



# Introduction to UNIX Streams and Pipes

![ghostbusters](https://i.giphy.com/media/v1.Y2lkPTc5MGI3NjExejhtamY1OHdoaXg3M253MGQxOXZ2and1aW05a3ZtaHRuc3hlYjR6NSZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/3o72FiKtrMAjIb0Rhu/giphy.gif)




---

## What Are UNIX Streams?

- **Streams** are a series of bytes of data that flow from one place to another.
- There are three standard streams:
  1. **Standard Input (stdin)**: Data coming into a program.
  2. **Standard Output (stdout)**: Data the program outputs.
  3. **Standard Error (stderr)**: Error messages from the program.

---




---

## Standard Streams Breakdown

- **stdin**: Usually from the keyboard but can be from files or other programs.
- **stdout**: Default is your terminal window, but it can be redirected elsewhere.
- **stderr**: Like stdout but meant for error messages, so it can be handled separately.

---



### Standard output example

`printf` writes predictable text to stdout and is available in Bash:

```bash
printf 'alpha\nbeta\n'
```


In [ ]:
%%bash
printf 'alpha\nbeta\n'


In [ ]:
%%bash




---
## What Are UNIX Pipes?

- **Pipes (`|`)**: A method of connecting the output of one command directly into the input of another command.
  
  - Example:
    ```bash
    command1 | command2
    ```
  - Output of `command1` becomes input of `command2`.

---



### A first pipeline

Pipe the output from `printf` into `grep`:

```bash
printf 'error: disk full\ninfo: complete\n' | grep '^error:'
```

The pipe sends stdout from `printf` to stdin for `grep`.


In [ ]:
%%bash
printf 'error: disk full\ninfo: complete\n' | grep '^error:'


In [ ]:
%%bash




---
## Benefits of Pipes

- **Modular Processing**: Each command in the pipeline does one job.
- **Efficiency**: Avoids creating temporary files.
- **Flexibility**: You can combine simple commands to perform complex tasks.

---



## Counting with a pipe: `find` into `wc -l`

A pipe sends the output of one command into the input of the next command.

Here we will use two commands:

- `find` lists files that match a rule;
- `wc -l` counts how many lines it receives.

Used together, they can answer: **how many matching files did `find` find?**


First, use `find` on its own. This searches from the current folder, `.`.

```bash
find . -name '*.txt' -type f
```

Read this as: find regular files, starting here, whose names end in `.txt`.


In [ ]:
%%bash
find . -name '*.txt' -type f


Now use `wc -l` on its own. It counts lines from whatever text it receives.

Here we give it three lines using `printf`:


In [ ]:
%%bash
printf 'first\nsecond\nthird\n' | wc -l


Now combine the ideas.

```bash
find . -name '*.txt' -type f | wc -l
```

The pipe, `|`, sends the list of filenames produced by `find` into `wc -l`. `wc -l` then counts the number of lines in that list. Because `find` prints one filename per line, this counts the number of matching `.txt` files.


In [ ]:
%%bash
find . -name '*.txt' -type f | wc -l


This is different from:

```bash
wc -l *.txt
```

That command counts the lines **inside each `.txt` file** in the current folder.

The pipeline below counts the **number of `.txt` filenames found**.


In [ ]:
%%bash
echo 'Lines inside each .txt file:'
wc -l *.txt

echo
echo 'Number of .txt files found by find:'
find . -name '*.txt' -type f | wc -l


## Passing filenames to another command

The previous pipeline counted filenames. Sometimes we want to do something **to the files themselves**, such as count the lines inside every `.txt` file found by `find`.

A common next step is `xargs`, which turns incoming text into command arguments:

```bash
find . -name '*.txt' -type f | xargs wc -l
```

This means: find the `.txt` files, then pass those filenames to `wc -l`.


In [ ]:
%%bash
find . -name '*.txt' -type f | xargs wc -l


## Why filename safety matters later

The plain `find ... | xargs ...` pattern is useful, but it can break if filenames contain spaces.

The next example creates a temporary folder with two files, including one called `field work.txt`. Watch how ordinary `xargs` splits that filename incorrectly.


In [ ]:
%%bash
DEMO_DIR=$(mktemp -d)
printf 'alpha\nbeta\n' > "$DEMO_DIR/simple.txt"
printf 'one\ntwo\nthree\n' > "$DEMO_DIR/field work.txt"

find "$DEMO_DIR" -name '*.txt' -type f | xargs wc -l || true


There are two safer forms worth recognising.

One is `find -exec`, where `find` passes the filenames directly to the command:

```bash
find folder -name '*.txt' -type f -exec wc -l {} +
```

The other is the null-delimited `find` and `xargs` form:

```bash
find folder -name '*.txt' -type f -print0 | xargs -0 wc -l
```

Here, `-print0` and `xargs -0` agree to separate filenames with a special null character rather than spaces or newlines. That keeps awkward filenames intact.


In [ ]:
%%bash
DEMO_DIR=$(mktemp -d)
printf 'alpha\nbeta\n' > "$DEMO_DIR/simple.txt"
printf 'one\ntwo\nthree\n' > "$DEMO_DIR/field work.txt"

echo 'Using find -exec:'
find "$DEMO_DIR" -name '*.txt' -type f -exec wc -l {} +

echo
echo 'Using find -print0 with xargs -0:'
find "$DEMO_DIR" -name '*.txt' -type f -print0 | xargs -0 wc -l


For now, keep the main pipe idea clear:

```bash
command 1 | command 2
```

The output of command 1 becomes the input of command 2.


## Redirecting streams

- `>` sends stdout to a file and overwrites that file.
- `>>` appends stdout to a file.
- `2>` sends stderr to a file.

Use a controlled failure so both streams can be inspected:

```bash
ls log.txt missing_file > listing.txt 2> error_log.txt
```


In [ ]:
%%bash
set +e
ls log.txt missing_file > listing.txt 2> error_log.txt
status=$?
printf 'exit status: %s\n' "$status"
printf '%s\n' '--- stdout ---'
cat listing.txt
printf '%s\n' '--- stderr ---'
cat error_log.txt


In [ ]:
%%bash





## Combining stdout and stderr

Send stdout and stderr to the same file:

```bash
ls log.txt missing_file > combined.log 2>&1
```

Redirections are processed from left to right. Here, file descriptor 2 is pointed at the same destination as file descriptor 1.


In [ ]:
%%bash
set +e
ls log.txt missing_file > combined.log 2>&1
cat combined.log


In [ ]:
%%bash



## Filters in pipes

Filters read input and produce transformed output. Examples include `grep`, `sort`, `cut`, `awk`, and `sed`.

`ps aux` reports CPU percentage in column 3 and memory percentage in column 4. This pipeline shows the five highest current CPU values:

```bash
ps aux | sort -nrk 3,3 | head -n 5
```


In [ ]:
%%bash
ps aux | sort -nrk 3,3 | head -n 5


In [ ]:
%%bash




---
## Summary

- **Streams**: stdin, stdout, stderr – standard communication channels.
- **Pipes**: Connect output of one command to the input of another.
- **Redirection**: Modify where input/output goes, even to files.
- **Filters**: Tools to manipulate data within pipes for flexible processing.


### Note: Most command line bioinformatics programs can be used with streams, pipes, redirection and filters.
---



# UNIX Text Processing Tools: grep, sort, cut, awk, and sed

---

## Introduction to `grep`

- **`grep`** is used to search for patterns within files.
- It stands for **global regular expression print**.
  
### Syntax:
```bash
grep [options] pattern [file...]
```

### Example:
```bash
grep "error" log.txt
```
Searches for the word "error" in log.txt.

---

---

## grep Options

-i: Case-insensitive search.

-v: Invert match (show lines that don't match the pattern).

-r: Search directories recursively.


Example:
```bash
grep -i "warning" log.txt
```

Case-insensitive search for "warning" in log.txt.

---

---

# Introduction to sort

sort is used to sort lines of text files.

Syntax:

```bash
sort [options] [file...]
```

Example:
```bash
sort data.txt
```

Sorts the contents of data.txt in alphabetical order.

---

## `sort` options

- `-r`: reverse order.
- `-n`: numeric comparison.
- `-t`: field delimiter.
- `-k`: key field.

The supplied `data.txt` is comma separated, so sort its age field with:

```bash
sort -t, -k2,2n data.txt
```


---

## Introduction to cut

cut is used to extract specific fields from files.

Syntax:
```bash
cut [options] [file...]
```

Example:
```bash
cut -d "," -f 1,3 data.csv
```

Extracts the 1st and 3rd columns from data.csv using , as a delimiter.

---

## `cut` options

- `-d`: field delimiter.
- `-f`: selected fields.

The supplied `data.txt` uses commas:

```bash
cut -d, -f2-3 data.txt
```


## Introduction to awk

awk is a powerful text processing tool, particularly for structured data.

Syntax:
```bash
awk 'pattern {action}' [file...]
```

Example:

```bash
awk '{print $1, $3}' data.txt
```

Prints the 1st and 3rd columns of each line from data.txt.

## `awk` with a delimiter and condition

Skip the header in `data.csv`, interpret commas as field separators, and report salaries above 5000:

```bash
awk -F, 'NR > 1 && $3 > 5000 {print $1, $3}' data.csv
```


## Introduction to sed

sed is a stream editor for filtering and transforming text.

Syntax:
```bash
sed 'command' [file...]
```

Example:
```bash
sed 's/error/ERROR/g' log.txt
```

Replaces all occurrences of "error" with "ERROR" in log.txt.



## `sed` options and portability

Substitution syntax is `s/pattern/replacement/`. Start with non-destructive output:

```bash
sed 's/warning/NOTICE/g' log.txt > revised_log.txt
```

GNU and BSD/macOS `sed -i` use different syntax. Avoid in-place editing in introductory cross-platform examples; write a new file and inspect it first.


## Combining tools with biological-style data

The supplied tab-separated `read_metrics.tsv` contains a header and PASS/FAIL records. List passing sample names and read counts, highest first:

```bash
grep $'\tPASS$' read_metrics.tsv | cut -f1,2 | sort -k2,2nr
```

Inspect the output after each stage while developing the pipeline.


## Summary

grep: Searches for patterns.

sort: Sorts lines.

cut: Extracts specific columns.

awk: Processes structured text with conditions.

sed: Edits and transforms text streams.




## Bash Examples for Each Tool



1. grep Examples:
```bash
# Basic usage to search for a pattern
grep "error" log.txt

# Case-insensitive search
grep -i "error" log.txt

# Search recursively through directories
grep -r "error" /var/logs/
```

2. `sort` examples:
```bash
# Sort alphabetically
sort data.txt

# Sort numerically
sort -n numbers.txt

# Sort comma-separated data by the second field
sort -t, -k2,2n data.txt
```


3. `cut` examples:

```bash
# Extract first and third fields from a CSV file
cut -d, -f1,3 data.csv

# Extract fields two and three from comma-separated data
cut -d, -f2-3 data.txt
```


4. `awk` examples:

```bash
# Print the first and third comma-separated fields
awk -F, '{print $1, $3}' data.txt

# Skip the header and filter salary
awk -F, 'NR > 1 && $3 > 5000 {print $1, $3}' data.csv
```


5. `sed` examples:

```bash
# Display transformed text without changing the input
sed 's/error/ERROR/g' log.txt

# Write the transformed result to a new file
sed 's/warning/NOTICE/g' log.txt > revised_log.txt
```


6. Combining tools:

```bash
grep $'\tPASS$' read_metrics.tsv | cut -f1,2 | sort -k2,2nr
```
